# Service discovery & health checking, plainly

Companion notebook for System Design, Plainly — Season 2, Part 8. Toy models, all computed locally: detection delay as wasted requests, lease-expiry Monte Carlo, availability arithmetic, and stale-entry probability. Field figures are cited in the post.

In [ ]:
"""Cell 1: detection delay is wasted requests.
Fleet: 100 instances, 10k req/s evenly spread -> 100 req/s per instance.
One instance dies; every request routed to it until detection removes it."""
req_per_s_per_instance = 10_000 / 100
delays = {'Consul gossip': 2, 'K8s readiness probe': 10, 'DNS TTL': 60, 'Eureka heartbeat': 90}
for name, d in delays.items():
    print(f'{name:22s} {d:>2}s delay -> {req_per_s_per_instance*d:>6,.0f} wasted requests')
assert req_per_s_per_instance * 90 / (req_per_s_per_instance * 2) == 45
print('OK: Eureka-style heartbeats waste 45x more requests than gossip per dead instance')

In [ ]:
"""Cell 2: lease-expiry Monte Carlo.
Eureka-style: heartbeat every 30 s, evict after 90 s without renewal.
Crash time is uniform inside the heartbeat interval, so detection is
90 s minus the time since the last heartbeat. Gossip (SWIM): ~2 s, near-constant."""
import numpy as np
rng = np.random.default_rng(7)
crash_offset = rng.uniform(0, 30, size=100_000)   # seconds since last heartbeat
eureka_detect = 90 - crash_offset
gossip_detect = rng.normal(2.0, 0.3, size=100_000).clip(min=0.5)
print(f'Eureka-style detection: mean {eureka_detect.mean():.1f}s, p99 {np.percentile(eureka_detect, 99):.1f}s')
print(f'Gossip detection:       mean {gossip_detect.mean():.2f}s, p99 {np.percentile(gossip_detect, 99):.2f}s')
assert 70 < eureka_detect.mean() < 80, 'mean should be ~75s (90 - 15)'
assert gossip_detect.mean() < 3
print('OK: heartbeat detection averages ~75 s - the eviction timeout, not the heartbeat, dominates')

In [ ]:
"""Cell 3: availability arithmetic - downtime minutes per year."""
def downtime_min(avail):
    return (1 - avail) * 365.25 * 24 * 60
for a in [0.99, 0.999, 0.9999, 0.99999]:
    print(f'{a*100:.3f}% availability -> {downtime_min(a):,.1f} min/yr downtime')
assert abs(downtime_min(0.99999) - 5.26) < 0.01
assert abs(downtime_min(0.999) - 525.96) < 0.1
# Registry at 99.9% with no client cache = system inherits 525.6 min/yr of discovery downtime.
# With a client-side cache + watch, registry downtime costs ~0 extra: lookups keep working.
print('OK: 5 nines = 5.26 min/yr; a 99.9% registry without client caching hands the system 526.0 min/yr')

In [ ]:
"""Cell 4: stale-entry probability under churn.
Fleet N=1000, churn 5 restarts/min. Dead entries linger for the detection delay d,
so expected stale entries = churn_rate * d, and a random lookup hits one with
probability stale / N (registry lists dead + live together until eviction)."""
N = 1000
churn_per_s = 5 / 60
for name, d in [('Consul gossip', 2), ('K8s probe', 10), ('DNS TTL', 60), ('Eureka', 90)]:
    stale = churn_per_s * d
    p = stale / N
    print(f'{name:16s} d={d:>2}s -> {stale:5.2f} stale entries, P(hit dead) = {p*100:.3f}%  ({p*1_000_000:,.0f} per 1M lookups)')
assert abs((churn_per_s * 90 / N) - 0.0075) < 1e-9
print('OK: at 90 s detection, 0.75% of lookups hit a dead entry - 7,500 per million')

In [ ]:
"""Cell 5: retry amplification - a dead entry + retries cascade.
Client retries a dead instance 3 times before trying the next one: 4x load on the
dead path, and if the retry budget is shared, the next-healthy-instance gets the
retried traffic too. Fail-fast + circuit breaker (Part 10) is the fix."""
retries = 3
wasted_per_request = 1 + retries
print(f'1 request to a dead entry with {retries} retries = {wasted_per_request} wasted attempts')
# Fleet-wide: if 0.75% of lookups hit dead entries (cell 4, Eureka) and each wastes 4 attempts,
# dead-path attempt share = 0.0075 * 4 = 3% of all attempts - 4x the stale-entry rate.
share = 0.0075 * wasted_per_request
print(f'Dead-path attempt share at 0.75% stale entries: {share*100:.1f}% of all attempts')
assert abs(share - 0.03) < 1e-9
print('OK: retries multiply the stale-entry cost - detect fast, fail fast, retry once')